In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip install torch torchvision --quiet
!pip install nibabel tqdm wandb --quiet


In [ ]:
# Step 2: Define dataset paths
DATA_ROOT = "/content/drive/MyDrive/total_segmentator_refine"

DATASETS = {
    "femur": f"{DATA_ROOT}/femur",
    "hip":   f"{DATA_ROOT}/hip",
    "rib":   f"{DATA_ROOT}/rib"
}

for name, path in DATASETS.items():
    print(f"{name.capitalize()} dataset -> {path}")


Femur dataset -> /content/drive/MyDrive/total_segmentator_refine/femur
Hip dataset -> /content/drive/MyDrive/total_segmentator_refine/hip
Rib dataset -> /content/drive/MyDrive/total_segmentator_refine/rib


In [ ]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
import os
from glob import glob
from PIL import Image

def interactive_ct_xray_viewer(dataset_root, patient_id="p_0001"):
    """
    Display AP/LAT X-rays with interactive CT slices (Coronal & Sagittal only).
    dataset_root: e.g. /content/drive/MyDrive/total_segmentator_refine/rib
    patient_id: e.g. p_0001
    """
    patient_dir = os.path.join(dataset_root, patient_id)

    # --- Load X-rays ---
    ap_path = glob(os.path.join(patient_dir, "x-ray", "*ap*.png"))
    lat_path = glob(os.path.join(patient_dir, "x-ray", "*lat*.png"))
    ap_img = Image.open(ap_path[0]).convert("L") if ap_path else None
    lat_img = Image.open(lat_path[0]).convert("L") if lat_path else None

    # --- Load CT ---
    vol_path = glob(os.path.join(patient_dir, "ct-scan", "*.nii.gz"))
    if not vol_path:
        print(f"⚠️ No CT file found in {patient_dir}")
        return

    nii = nib.load(vol_path[0])
    volume = nii.get_fdata()
    volume = (volume - volume.min()) / (volume.max() - volume.min())  # normalize [0,1]

    # --- Widgets ---
    slice_slider = widgets.IntSlider(min=0, max=volume.shape[0]-1, step=1, description='Slice:')
    axis_selector = widgets.Dropdown(
        options=['Coronal', 'Sagittal'],  # ✅ only these two
        value='Coronal',
        description='View:'
    )

    # --- Update function ---
    def view_slice(slice_index, view_axis):
        fig, axes = plt.subplots(1, 3 if lat_img else 2, figsize=(15,5))

        # AP X-ray
        if ap_img:
            axes[0].imshow(ap_img, cmap="gray")
            axes[0].set_title("AP X-ray")
            axes[0].axis("off")

        # LAT X-ray
        if lat_img:
            axes[1].imshow(lat_img, cmap="gray")
            axes[1].set_title("LAT X-ray")
            axes[1].axis("off")

        # CT slice (restricted to Coronal/Sagittal)
        if view_axis == 'Coronal':
            ct_slice = volume[:, slice_index, :]
        else:  # Sagittal
            ct_slice = volume[:, :, slice_index]

        axes[-1].imshow(ct_slice.T, cmap="gray", origin="lower")
        axes[-1].set_title(f"{view_axis} CT - Slice {slice_index}")
        axes[-1].axis("off")

        plt.show()

    display(widgets.interact(view_slice, slice_index=slice_slider, view_axis=axis_selector))


In [ ]:
DATA_ROOT = "/content/drive/MyDrive/total_segmentator_refine"

# Rib
interactive_ct_xray_viewer(os.path.join(DATA_ROOT, "rib"), patient_id="p_0001")

# Femur
interactive_ct_xray_viewer(os.path.join(DATA_ROOT, "femur"), patient_id="p_0001")

# Hip
interactive_ct_xray_viewer(os.path.join(DATA_ROOT, "hip"), patient_id="p_0001")


interactive(children=(IntSlider(value=0, description='Slice:', max=212), Dropdown(description='View:', options…

<function __main__.interactive_ct_xray_viewer.<locals>.view_slice(slice_index, view_axis)>

interactive(children=(IntSlider(value=0, description='Slice:', max=84), Dropdown(description='View:', options=…

<function __main__.interactive_ct_xray_viewer.<locals>.view_slice(slice_index, view_axis)>

interactive(children=(IntSlider(value=0, description='Slice:', max=191), Dropdown(description='View:', options…

<function __main__.interactive_ct_xray_viewer.<locals>.view_slice(slice_index, view_axis)>

In [ ]:
# Step 3: Import Libraries

# Core
import os
import numpy as np
import random

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# Torchvision (for transforms, image utilities)
import torchvision.transforms as transforms

# Medical imaging
import nibabel as nib  # for NIfTI (.nii.gz) CT scans

# Visualization
import matplotlib.pyplot as plt
import cv2  # for handling PNG X-rays
from tqdm import tqdm  # for progress bars
from scipy.ndimage import zoom

# Experiment tracking
import wandb


In [ ]:
class BoneDataset(Dataset):
    def __init__(self, root_dir, prefix, transform_2d=None, target_shape=(64, 64, 64)):
        self.root_dir = root_dir
        self.prefix = prefix
        self.transform_2d = transform_2d
        self.target_shape = target_shape
        self.patient_dirs = sorted(glob.glob(os.path.join(root_dir, "p_*")))

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]

        # Look for X-rays (be more flexible with file names)
        ap_path = glob.glob(os.path.join(patient_dir, "x-ray", f"*ap*.png"))
        lat_path = glob.glob(os.path.join(patient_dir, "x-ray", f"*lat*.png"))
        vol_path = glob.glob(os.path.join(patient_dir, "ct-scan", "*.nii.gz"))

        if not ap_path or not vol_path:
            raise FileNotFoundError(f"Missing AP/CT file in {patient_dir}")

        # Load X-rays
        ap_img = Image.open(ap_path[0]).convert("L")
        lat_img = None
        if lat_path:
            lat_img = Image.open(lat_path[0]).convert("L")

        # Apply transforms
        if self.transform_2d:
            ap_img = self.transform_2d(ap_img)
            if lat_img:
                lat_img = self.transform_2d(lat_img)

        # Load CT
        volume = nib.load(vol_path[0]).get_fdata()
        resize_factor = [t / s for t, s in zip(self.target_shape, volume.shape)]
        volume = zoom(volume, resize_factor, order=1)
        volume = torch.from_numpy(volume).float().unsqueeze(0)  # [1, D, H, W]

        return ap_img, lat_img, volume

In [ ]:
class SmallBoneDataset(Dataset):
    def __init__(self, root_dir, transform_2d=None, target_shape=(64, 64, 64)):
        self.root_dir = root_dir
        self.transform_2d = transform_2d
        self.target_shape = target_shape
        self.patient_dirs = sorted(glob.glob(os.path.join(root_dir, "CLINIC_*")))

    def __len__(self):
        return len(self.patient_dirs)

    def __getitem__(self, idx):
        patient_dir = self.patient_dirs[idx]

        # --- File paths ---
        ap_path = glob.glob(os.path.join(patient_dir, "derivatives", "xray_from_ct", "*ap*.png"))
        lat_path = glob.glob(os.path.join(patient_dir, "derivatives", "xray_from_ct", "*lat*.png"))
        vol_path = glob.glob(os.path.join(patient_dir, "derivatives", "seg_roi", "*.nii.gz"))

        if not ap_path or not lat_path or not vol_path:
            raise FileNotFoundError(f"Missing AP/LAT/CT file in {patient_dir}")

        # --- Load X-rays ---
        ap_img = Image.open(ap_path[0]).convert("L")
        lat_img = Image.open(lat_path[0]).convert("L")

        # --- Apply transforms ---
        if self.transform_2d:
            ap_img = self.transform_2d(ap_img)
            lat_img = self.transform_2d(lat_img)
        if ap_img.ndim == 2: ap_img = ap_img.unsqueeze(0)
        if lat_img.ndim == 2: lat_img = lat_img.unsqueeze(0)
        ap_img = ap_img.float() / 255.0
        lat_img = lat_img.float() / 255.0

        # --- Load CT ---
        volume = nib.load(vol_path[0]).get_fdata().astype(np.float32)
        volume = (volume - volume.min()) / (volume.max() - volume.min() + 1e-8)
        if volume.shape[-1] != self.target_shape[0]:  # optional axis check
            volume = np.transpose(volume, (2, 0, 1))  # [D,H,W]
        resize_factor = [t / s for t, s in zip(self.target_shape, volume.shape)]
        volume = zoom(volume, resize_factor, order=1)
        volume = torch.from_numpy(volume).float().unsqueeze(0)  # [1, D, H, W]

        return ap_img, lat_img, volume


NameError: name 'Dataset' is not defined

In [ ]:
# Common transform for X-rays
xray_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
])

# Dataset root
DATA_ROOT = "/content/drive/MyDrive/total_segmentator_refine"

# Initialize datasets
datasets = {
    "rib": BoneDataset(os.path.join(DATA_ROOT, "rib"), prefix="rib", transform_2d=xray_transform),
    "femur": BoneDataset(os.path.join(DATA_ROOT, "femur"), prefix="femur", transform_2d=xray_transform),
    "hip": BoneDataset(os.path.join(DATA_ROOT, "hip"), prefix="hip", transform_2d=xray_transform),
}

# Check sample
ap, lat, vol = datasets["rib"][0]
print("Rib sample:", ap.shape, lat.shape if lat is not None else None, vol.shape)

ap, lat, vol = datasets["femur"][0]
print("Femur sample:", ap.shape, lat.shape if lat is not None else None, vol.shape)

ap, lat, vol = datasets["hip"][0]
print("Hip sample:", ap.shape, lat.shape if lat is not None else None, vol.shape)

Rib sample: torch.Size([1, 128, 128]) torch.Size([1, 128, 128]) torch.Size([1, 64, 64, 64])
Femur sample: torch.Size([1, 128, 128]) torch.Size([1, 128, 128]) torch.Size([1, 64, 64, 64])
Hip sample: torch.Size([1, 128, 128]) torch.Size([1, 128, 128]) torch.Size([1, 64, 64, 64])


In [ ]:
from torchvision import transforms

transform_2d = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor()
])

small_dataset = SmallBoneDataset(
    root_dir="/content/drive/MyDrive/2dto3d/BIDS",
    transform_2d=transform_2d,
    target_shape=(64,64,64)
)

print("Total patients:", len(small_dataset))

ap, lat, vol = small_dataset[0]
print("AP shape :", ap.shape)
print("LAT shape:", lat.shape if lat is not None else None)
print("CT shape :", vol.shape)


Total patients: 103
AP shape : torch.Size([1, 128, 128])
LAT shape: torch.Size([1, 128, 128])
CT shape : torch.Size([1, 64, 64, 64])


In [ ]:
from torch.utils.data import random_split, DataLoader

# Check dataset size
small_total_size = len(small_dataset)
print("Total patients:", small_total_size)

# Define split sizes (70/20/10 split for example)
small_train_size = int(0.7 * small_total_size)
small_val_size   = int(0.2 * small_total_size)
small_test_size  = small_total_size - small_train_size - small_val_size

print(f"Train: {small_train_size}, Val: {small_val_size}, Test: {small_test_size}")

# Split dataset (with fixed seed for reproducibility)
small_generator = torch.Generator().manual_seed(42)
small_train_set, small_val_set, small_test_set = random_split(
    small_dataset, [small_train_size, small_val_size, small_test_size], generator=small_generator
)

# Create DataLoaders
small_train_loader = DataLoader(small_train_set, batch_size=2, shuffle=True, num_workers=2, pin_memory=True)
small_val_loader   = DataLoader(small_val_set, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)
small_test_loader  = DataLoader(small_test_set, batch_size=2, shuffle=False, num_workers=2, pin_memory=True)

# Inspect one batch
for ap, lat, vol in small_train_loader:
    print("AP batch :", ap.shape)
    print("LAT batch:", lat.shape)
    print("CT batch :", vol.shape)
    break


Total patients: 103
Train: 72, Val: 20, Test: 11
AP batch : torch.Size([2, 1, 128, 128])
LAT batch: torch.Size([2, 1, 128, 128])
CT batch : torch.Size([2, 1, 64, 64, 64])


In [ ]:
from torch.utils.data import ConcatDataset

# Combine rib, femur, hip into one dataset
combined_dataset = ConcatDataset([
    datasets["rib"],
    datasets["femur"],
    datasets["hip"]
])

print("Total samples in combined dataset:", len(combined_dataset))


Total samples in combined dataset: 1858


In [ ]:
from torch.utils.data import random_split, DataLoader

def create_dataloaders(dataset, batch_size=2, val_split=0.2, test_split=0.1):
    total_size = len(dataset)
    val_size = int(val_split * total_size)
    test_size = int(test_split * total_size)
    train_size = total_size - val_size - test_size

    train_set, val_set, test_set = random_split(
        dataset, [train_size, val_size, test_size],
        generator=torch.Generator().manual_seed(42)
    )

    loaders = {
        "train": DataLoader(train_set, batch_size=batch_size, shuffle=True),
        "val": DataLoader(val_set, batch_size=batch_size, shuffle=False),
        "test": DataLoader(test_set, batch_size=batch_size, shuffle=False),
    }
    return loaders

# Create loaders
loaders = create_dataloaders(combined_dataset, batch_size=2)

print("Train size:", len(loaders["train"].dataset))
print("Val size:", len(loaders["val"].dataset))
print("Test size:", len(loaders["test"].dataset))

Train size: 1302
Val size: 371
Test size: 185


In [ ]:
ap, lat, vol = next(iter(loaders["train"]))
print("AP batch shape:", ap.shape)
print("LAT batch shape:", lat.shape if lat is not None else None)
print("CT batch shape:", vol.shape)


AP batch shape: torch.Size([2, 1, 128, 128])
LAT batch shape: torch.Size([2, 1, 128, 128])
CT batch shape: torch.Size([2, 1, 64, 64, 64])


In [ ]:
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

def interactive_viewer(ap, lat, vol, idx=0):
    ap_img = ap[idx].squeeze().cpu().numpy()
    lat_img = lat[idx].squeeze().cpu().numpy() if lat is not None else None
    vol_np = vol[idx].squeeze().cpu().numpy()

    D, H, W = vol_np.shape

    # Dropdown for view selection (Sagittal / Coronal)
    view_dropdown = widgets.Dropdown(
        options=["Sagittal", "Coronal"],
        value="Sagittal",
        description="View:"
    )

    # Slider (max depends on view)
    slice_slider = widgets.IntSlider(min=0, max=W-1, step=1, value=W//2, description="Slice:")

    def update_plot(view, slice_idx):
        plt.figure(figsize=(15,5))

        # AP X-ray
        plt.subplot(1,3,1)
        plt.imshow(ap_img, cmap="gray")
        plt.title("AP X-ray")
        plt.axis("off")

        # LAT X-ray (if exists)
        if lat_img is not None:
            plt.subplot(1,3,2)
            plt.imshow(lat_img, cmap="gray")
            plt.title("LAT X-ray")
            plt.axis("off")

        # CT Sagittal / Coronal
        plt.subplot(1,3,3)
        if view == "Sagittal":
            plt.imshow(vol_np[:, :, slice_idx], cmap="gray")
            plt.title(f"Sagittal slice {slice_idx}/{W-1}")
        else:  # Coronal
            plt.imshow(vol_np[:, slice_idx, :], cmap="gray")
            plt.title(f"Coronal slice {slice_idx}/{H-1}")
        plt.axis("off")

        plt.show()

    # Interactive link
    ui = widgets.VBox([view_dropdown, slice_slider])
    out = widgets.interactive_output(update_plot, {"view": view_dropdown, "slice_idx": slice_slider})

    display(ui, out)

# 🔹 Usage
ap_batch, lat_batch, vol_batch = next(iter(loaders["train"]))
interactive_viewer(ap_batch, lat_batch, vol_batch, idx=0)


Output()

In [ ]:
# Import
import wandb

# Login with your API key (keep this private!)
!wandb login 47db746f38dd6216f60f4b6ce9c5d98e507dd6a3

wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [ ]:
import torch
import torch.nn as nn

# -------------------------------
# Encoder for Sagittal View
# -------------------------------
class EncoderSagittal(nn.Module):
    """
    Input:  [B,1,128,128]
    Output: [B,32,1,16,16]
    """
    def __init__(self, in_channels=1, out_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1)), # 128→64
            nn.ReLU(inplace=True),
            nn.Conv3d(16, out_channels, kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1)), # 64→32
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=(1,3,3), stride=(1,2,2), padding=(0,1,1)), # 32→16
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = x.unsqueeze(2)  # [B,1,1,128,128]
        return self.encoder(x)  # [B,32,1,16,16]


# -------------------------------
# Encoder for Coronal View
# -------------------------------
class EncoderCoronal(nn.Module):
    """
    Input:  [B,1,128,128]
    Output: [B,32,16,16,1]
    """
    def __init__(self, in_channels=1, out_channels=32):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=(3,3,1), stride=(2,2,1), padding=(1,1,0)), # 128→64
            nn.ReLU(inplace=True),
            nn.Conv3d(16, out_channels, kernel_size=(3,3,1), stride=(2,2,1), padding=(1,1,0)), # 64→32
            nn.ReLU(inplace=True),
            nn.Conv3d(out_channels, out_channels, kernel_size=(3,3,1), stride=(2,2,1), padding=(1,1,0)), # 32→16
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        x = x.unsqueeze(4)  # [B,1,128,128,1]
        return self.encoder(x)  # [B,32,16,16,1]


# -------------------------------
# Depth Expander (Sagittal Path)
# -------------------------------
class DepthExpander(nn.Module):
    """ Expand sagittal depth 1→16 """
    def __init__(self, out_depth=16):
        super().__init__()
        self.out_depth = out_depth
        self.expand = nn.ConvTranspose3d(32, 32, kernel_size=(2,1,1), stride=(2,1,1))
        self.refine = nn.Sequential(
            nn.Conv3d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv3d(32, 32, 3, padding=1), nn.ReLU()
        )

    def forward(self, x):
        while x.shape[2] < self.out_depth:
            x = self.expand(x)
        x = x[:,:,:self.out_depth,:,:]
        return self.refine(x)  # [B,32,16,16,16]


# -------------------------------
# Width Expander (Coronal Path)
# -------------------------------
class WidthExpander(nn.Module):
    """ Expand coronal width 1→16 """
    def __init__(self, out_width=16):
        super().__init__()
        self.out_width = out_width
        self.expand = nn.ConvTranspose3d(32, 32, kernel_size=(1,1,2), stride=(1,1,2))
        self.refine = nn.Sequential(
            nn.Conv3d(32, 32, 3, padding=1), nn.ReLU(),
            nn.Conv3d(32, 32, 3, padding=1), nn.ReLU()
        )

    def forward(self, x):
        while x.shape[4] < self.out_width:
            x = self.expand(x)
        x = x[:,:,:,:, :self.out_width]
        return self.refine(x)  # [B,32,16,16,16]


# -------------------------------
# 3D Volume Decoder
# -------------------------------
class Volume3DDecoder(nn.Module):
    """
    Input:  [B,64,16,16,16]
    Output: [B,1,64,64,64]
    """
    def __init__(self, out_channels=1):
        super().__init__()
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(64,64,4,2,1), nn.ReLU(), nn.BatchNorm3d(64),  # 16→32
            nn.ConvTranspose3d(64,32,4,2,1), nn.ReLU(), nn.BatchNorm3d(32),  # 32→64
            nn.Conv3d(32,32,3,padding=1), nn.ReLU(),
            nn.Conv3d(32,out_channels,3,padding=1)
        )

    def forward(self,x):
        return self.decoder(x)


# -------------------------------
# Full Model
# -------------------------------
class TransVert(nn.Module):
    def __init__(self,in_channels=1,out_channels=1,intermediate_depth=16):
        super().__init__()
        self.sagittal_encoder=EncoderSagittal(in_channels)
        self.coronal_encoder=EncoderCoronal(in_channels)
        self.depth_expander=DepthExpander(intermediate_depth)
        self.width_expander=WidthExpander(intermediate_depth)
        self.decoder=Volume3DDecoder(out_channels)

    def forward(self,sagittal_input,coronal_input):
        sag=self.sagittal_encoder(sagittal_input)   # [B,32,1,16,16]
        cor=self.coronal_encoder(coronal_input)     # [B,32,16,16,1]
        sag=self.depth_expander(sag)                # [B,32,16,16,16]
        cor=self.width_expander(cor)                # [B,32,16,16,16]
        fused=torch.cat([sag,cor],dim=1)            # [B,64,16,16,16]
        return self.decoder(fused)                  # [B,1,64,64,64]


In [ ]:
# Instantiate model
model = TransVert(in_channels=1, out_channels=1)

# Dummy AP/LAT inputs
ap = torch.randn(2,1,128,128)
lat = torch.randn(2,1,128,128)

with torch.no_grad():
    out = model(ap, lat)

print("AP batch shape:", ap.shape)
print("LAT batch shape:", lat.shape)
print("CT batch shape:", out.shape)

AP batch shape: torch.Size([2, 1, 128, 128])
LAT batch shape: torch.Size([2, 1, 128, 128])
CT batch shape: torch.Size([2, 1, 64, 64, 64])


In [ ]:
# =========================
# Loss Function (BCE + Dice)
# =========================
class DiceBCELoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.smooth = smooth

    def forward(self, preds, targets):
        bce_loss = self.bce(preds, targets)

        preds = torch.sigmoid(preds)
        preds_flat = preds.view(preds.size(0), -1)
        targets_flat = targets.view(targets.size(0), -1)

        intersection = (preds_flat * targets_flat).sum(1)
        dice_score = (2. * intersection + self.smooth) / (
            preds_flat.sum(1) + targets_flat.sum(1) + self.smooth
        )
        dice_loss = 1 - dice_score.mean()

        return bce_loss + dice_loss   # ✅ only loss returned


# =========================
# Metric Function (Dice)
# =========================
def compute_dice(preds, targets, smooth=1.0):
    preds = torch.sigmoid(preds)
    preds = (preds > 0.5).float()

    preds_flat = preds.view(preds.size(0), -1)
    targets_flat = targets.view(targets.size(0), -1)

    intersection = (preds_flat * targets_flat).sum(1)
    dice_score = (2. * intersection + smooth) / (
        preds_flat.sum(1) + targets_flat.sum(1) + smooth
    )
    return dice_score.mean().item()


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from tqdm import tqdm
import wandb
import os


# =========================
# Device & Config
# =========================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epochs = 20
lr = 1e-4
batch_size = 2
grad_clip = 1.0
patience = 10   # early stopping patience

save_path = "/content/drive/MyDrive/saved_models/best_model_01.pth"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

use_wandb = True
if use_wandb:
    wandb.init(project="TransvertFinal", config={
        "epochs": num_epochs,
        "lr": lr,
        "loss": "BCE+Dice",
        "batch_size": batch_size
    })

# =========================
# Model, Loss, Optimizer
# =========================
model = TransVert(in_channels=1, out_channels=1).to(device)
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=5)

# =========================
# Tracking
# =========================
train_epoch_losses, val_epoch_losses, val_dice_scores = [], [], []
best_val_loss = float("inf")
early_stop_counter = 0

# =========================
# Training & Validation
# =========================
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for sagittal, coronal, target in tqdm(loaders["train"], desc=f"Epoch {epoch+1}/{num_epochs}"):
        sagittal, coronal, target = sagittal.to(device), coronal.to(device), target.to(device)

        optimizer.zero_grad()
        output = model(sagittal, coronal)

        loss = criterion(output, target)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()

        running_loss += loss.item()

        if use_wandb:
            wandb.log({"train/batch_loss": loss.item()})

    avg_train_loss = running_loss / len(small_train_loader)
    train_epoch_losses.append(avg_train_loss)

    # =========================
    # Validation
    # =========================
    model.eval()
    val_loss, val_dice = 0.0, 0.0
    with torch.no_grad():
        for sagittal, coronal, target in loaders["val"]:
            sagittal, coronal, target = sagittal.to(device), coronal.to(device), target.to(device)
            output = model(sagittal, coronal)

            loss = criterion(output, target)
            dice = compute_dice(output, target)

            val_loss += loss.item()
            val_dice += dice

            if use_wandb:
                wandb.log({"val/batch_loss": loss.item(), "val/batch_dice": dice})

    avg_val_loss = val_loss / len(small_val_loader)
    avg_val_dice = val_dice / len(small_val_loader)
    val_epoch_losses.append(avg_val_loss)
    val_dice_scores.append(avg_val_dice)

    # Scheduler step
    scheduler.step(avg_val_loss)

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        early_stop_counter = 0
        torch.save(model.state_dict(), save_path)
        print(f"✅ Saved new best model at {save_path}")
    else:
        early_stop_counter += 1

    # Log epoch metrics
    if use_wandb:
        wandb.log({
            "epoch": epoch + 1,
            "train/epoch_loss": avg_train_loss,
            "val/epoch_loss": avg_val_loss,
            "val/epoch_dice": avg_val_dice,
            "learning_rate": optimizer.param_groups[0]['lr']
        })

    print(f"[Epoch {epoch+1}] Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Dice: {avg_val_dice:.4f}")

    # Early stopping
    if early_stop_counter >= patience:
        print(f"⏹️ Early stopping at epoch {epoch+1}")
        break

# =========================
# Testing
# =========================
print("Loading best model for testing...")
model.load_state_dict(torch.load(save_path))
model.eval()

test_loss, test_dice = 0.0, 0.0
with torch.no_grad():
    for sagittal, coronal, target in loaders["test"]:
        sagittal, coronal, target = sagittal.to(device), coronal.to(device), target.to(device)
        output = model(sagittal, coronal)

        loss = criterion(output, target)
        dice = compute_dice(output, target)

        test_loss += loss.item()
        test_dice += dice

avg_test_loss = test_loss / len(small_test_loader)
avg_test_dice = test_dice / len(small_test_loader)

print(f"📊 Test Loss: {avg_test_loss:.4f} | Test Dice: {avg_test_dice:.4f}")

if use_wandb:
    wandb.log({"test/loss": avg_test_loss, "test/dice": avg_test_dice})
    wandb.finish()

# =========================
# Plot Loss Curves
# =========================
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.plot(train_epoch_losses, label='Train Loss')
plt.plot(val_epoch_losses, label='Val Loss')
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Epoch-wise Loss")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(val_dice_scores, label='Val Dice', color='green')
plt.xlabel("Epoch")
plt.ylabel("Dice Score")
plt.title("Validation Dice Score")
plt.legend()

plt.tight_layout()
plt.show()


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Currently logged in as: jeevan-neupane003 (jeevan-neupane003-khalti) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch 1/20: 100%|██████████| 651/651 [43:52<00:00,  4.04s/it]


✅ Saved new best model at /content/drive/MyDrive/saved_models/best_model_01.pth
[Epoch 1] Train Loss: 15.9988 | Val Loss: 15.5215 | Val Dice: 6.0035


Epoch 2/20: 100%|██████████| 651/651 [02:46<00:00,  3.91it/s]


✅ Saved new best model at /content/drive/MyDrive/saved_models/best_model_01.pth
[Epoch 2] Train Loss: 12.6336 | Val Loss: 12.6137 | Val Dice: 8.7942


Epoch 3/20: 100%|██████████| 651/651 [02:47<00:00,  3.88it/s]


... (31 outputs left out)

Epoch 19/20: 100%|██████████| 651/651 [02:49<00:00,  3.84it/s]


[Epoch 19] Train Loss: 7.9739 | Val Loss: 9.3091 | Val Dice: 11.0789


Epoch 20/20: 100%|██████████| 651/651 [02:48<00:00,  3.86it/s]


✅ Saved new best model at /content/drive/MyDrive/saved_models/best_model_01.pth
[Epoch 20] Train Loss: 7.8971 | Val Loss: 9.2435 | Val Dice: 11.1140
Loading best model for testing...


In [ ]:
import torch
from torchvision import transforms
from PIL import Image

# Load model (assuming already defined as TransVert)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransVert(in_channels=1, out_channels=1).to(device)
model.load_state_dict(torch.load("/content/drive/MyDrive/saved_models/best_model_test_01.pth", map_location=device))
model.eval()

# Preprocessing
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

# Example: load 2 X-ray images from drive
ap_img = Image.open("/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/x-ray/femur_right-ap.png").convert("L")
lat_img = Image.open("/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/x-ray/femur_right-lat.png").convert("L")

ap_tensor = transform(ap_img).unsqueeze(0).to(device)   # [1,1,128,128]
lat_tensor = transform(lat_img).unsqueeze(0).to(device) # [1,1,128,128]

with torch.no_grad():
    output = model(ap_tensor, lat_tensor)  # [1,1,64,64,64]
    output = torch.sigmoid(output).cpu().numpy().squeeze()
print("Output volume shape:", output.shape)


In [ ]:
import matplotlib.pyplot as plt

def show_slices(volume, idx=None):
    if idx is None:
        idx = (volume.shape[0]//2, volume.shape[1]//2, volume.shape[2]//2)

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(volume[idx[0], :, :], cmap="gray")
    axes[0].set_title(f"Axial Slice {idx[0]}")
    axes[1].imshow(volume[:, idx[1], :], cmap="gray")
    axes[1].set_title(f"Coronal Slice {idx[1]}")
    axes[2].imshow(volume[:, :, idx[2]], cmap="gray")
    axes[2].set_title(f"Sagittal Slice {idx[2]}")
    plt.show()

show_slices(output)


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import numpy as np

def visualize_3d(volume, threshold=0.5):
    # Convert soft probabilities → binary mask
    vol_bin = (volume > threshold).astype(np.uint8)

    # Create coordinates
    x, y, z = np.indices(vol_bin.shape)

    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(111, projection="3d")

    # Voxels where bone exists
    ax.voxels(vol_bin, facecolors="white", edgecolors="gray", alpha=0.7)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Z")
    ax.set_title("3D Bone Reconstruction")
    plt.show()

# Example usage with your output
visualize_3d(output, threshold=0.3)


In [ ]:
import nibabel as nib
import numpy as np
import plotly.graph_objects as go

def render_ct_plotly(nii_path, threshold=0.5):
    vol = nib.load(nii_path).get_fdata()
    vol = (vol - vol.min()) / (vol.max() - vol.min())

    fig = go.Figure(data=go.Isosurface(
        x=np.arange(vol.shape[0]).repeat(vol.shape[1]*vol.shape[2]),
        y=np.tile(np.arange(vol.shape[1]).repeat(vol.shape[2]), vol.shape[0]),
        z=np.tile(np.arange(vol.shape[2]), vol.shape[0]*vol.shape[1]),
        value=vol.flatten(),
        isomin=threshold,
        isomax=1.0,
        surface_count=1,
        colorscale="Gray"
    ))

    fig.update_layout(scene=dict(
        xaxis_title="X", yaxis_title="Y", zaxis_title="Z"
    ))
    fig.show()

# Example usage
render_ct_plotly("/content/drive/MyDrive/total_segmentator_refine/femur/p_0001/ct-scan/femur_right_msk.nii.gz")
